In [5]:
import pandas as pd
import spacy
from spacy.pipeline import EntityRuler
from tqdm import tqdm

# Enable progress bar for pandas
tqdm.pandas()

# STEP 1: Load brand + model CSV
df_brands = pd.read_csv("./brands/brands.csv", sep=";", header=None, names=["brand", "model", "start_year", "end_year"])
df_brands = df_brands[["brand", "model"]]

# Create lowercase patterns for EntityRuler
patterns = df_brands.apply(
    lambda row: {"label": "CAR_MODEL", "pattern": (row["brand"] + " " + row["model"]).lower()},
    axis=1
).tolist()

# STEP 2: Setup SpaCy and EntityRuler
nlp = spacy.load("it_core_news_sm")
nlp.add_pipe("entity_ruler", before="ner")
ruler = nlp.get_pipe("entity_ruler")
ruler.add_patterns(patterns)

# STEP 3: Load dataset with titles
df = pd.read_csv("dataProcessed.csv")

# Get name of the first column (assumed to contain titles)
title_column = df.columns[0]

# STEP 4: Define extraction function
def extract_model(text):
    doc = nlp(str(text).lower())  # Lowercase input
    for ent in doc.ents:
        if ent.label_ == "CAR_MODEL":
            parts = ent.text.strip().split()
            if len(parts) >= 2:
                return pd.Series([parts[0], " ".join(parts[1:])])
    return pd.Series([None, None])

# STEP 5: Apply with progress bar
df[["brand", "model"]] = df[title_column].progress_apply(extract_model)

# STEP 6: Save result
df.to_csv("dataWithCarModel.csv", index=False)

100%|██████████| 109898/109898 [05:20<00:00, 343.41it/s]


In [7]:
# Count how many rows have missing brand or model
missing_brand = df["brand"].isna().sum()
missing_model = df["model"].isna().sum()

# Optional: rows where both are missing
missing_both = df[df["brand"].isna() & df["model"].isna()].shape[0]

print(f"⚠️ Missing brand: {missing_brand}")
print(f"⚠️ Missing model: {missing_model}")
print(f"⚠️ Rows with both missing: {missing_both}")

# Total unusable rows (at least one missing)
unusable_rows = df[df["brand"].isna() | df["model"].isna()].shape[0]
print(f"❌ Total rows without usable car model info: {unusable_rows}")
df.shape 

⚠️ Missing brand: 25257
⚠️ Missing model: 25257
⚠️ Rows with both missing: 25257
❌ Total rows without usable car model info: 25257


(109898, 11)

In [ ]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

# 1. Inizializza il client

# 2. Leggi i dati
df = pd.read_csv("dataWithCarModel.csv")
missing_df = df[df["brand"].isna() | df["model"].isna()].copy()

# 3. Funzione per estrarre brand e model
def ask_chatgpt(text):
    prompt = (
        f"Extract only the car brand and model from this title, "
        f"separate them with a semicolon: '{text}'"
    )
    try:
        resp = client.chat.completions.create(
            model="gpt-4.1-nano",
            messages=[
                {"role": "system", "content": "You are a car-extraction assistant."},
                {"role": "user", "content": prompt}
            ],
            temperature=0,
            max_tokens=50
        )
        result = resp.choices[0].message.content.strip()
        if ";" in result:
            brand, model = [p.strip() for p in result.split(";", 1)]
            print("✅", text, "->", brand, model)
            return pd.Series([brand, model])
        print("❌ failed:", text, "->", result)
        return pd.Series([None, None])
    except Exception as e:
        print("⚠️ API error:", e)
        return pd.Series([None, None])

# 4. Applica con barra di progresso
tqdm.pandas()
missing_df[["brand", "model"]] = missing_df["brand_model"].progress_apply(ask_chatgpt)

# 5. Aggiorna e salva
df.update(missing_df[["brand", "model"]])
df.to_csv("dataWithCarModel_ChatGPT_Nano.csv", index=False)

  0%|          | 2/25257 [00:01<4:00:09,  1.75it/s]

✅ Toyota LJ70VX -> Toyota LJ70VX


  0%|          | 3/25257 [00:02<5:07:07,  1.37it/s]

✅ Range Rover Evoque E6 2.0 VALUTIAMO USATO / -> Range Rover Evoque


  0%|          | 4/25257 [00:05<12:46:50,  1.82s/it]

...

100%|█████████▉| 25256/25257 [3:07:30<00:00,  2.43it/s]

✅ Mercedes-Benz GLA 180 d Automatic Sport Plus -> Mercedes-Benz GLA 180 d


100%|██████████| 25257/25257 [3:07:31<00:00,  2.44it/s]

✅ Peugeut 2008 -> Peugeot 2008
✅ Audi a 5 sportback quattro tfsi -> Audi A5 Sportback Quattro TFSI


100%|██████████| 25257/25257 [3:07:32<00:00,  2.24it/s]
